# Local Document Summarizer

**Fully local — no API key, no internet needed after setup.**

Point this notebook at any folder of documents and it will summarize each one using `llama3.2` running locally via Ollama.

**Supported file types:** `.txt`, `.md`, `.pdf`

**Pipeline:**
```
docs folder  →  read file  →  chunk if too long  →  llama3.2 (local)  →  markdown summary
```

### Prerequisites
- Ollama installed and `ollama serve` running
- `llama3.2` pulled: `ollama pull llama3.2`
- Kernel set to `udemy_env`

In [ ]:
# Install PDF support (one-time)
%pip install -q pypdf

In [ ]:
import requests

try:
    r = requests.get("http://localhost:11434")
    print(r.text)  # should say: Ollama is running
except Exception as e:
    print(f"Ollama not reachable: {e}")
    print("Run 'ollama serve' in a terminal first.")

In [ ]:
from pathlib import Path
from openai import OpenAI
from IPython.display import Markdown, display
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

In [ ]:
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = "llama3.2"

# llama3.2 handles ~4K characters comfortably per call
CHUNK_SIZE = 4000
CHUNK_OVERLAP = 200

## Set your docs folder

Change `DOCS_FOLDER` to point at the folder you want to summarize.

In [ ]:
DOCS_FOLDER = Path("/path/to/your/docs")  # <-- change this

# Check it exists and show what's inside
if not DOCS_FOLDER.exists():
    print(f"Folder not found: {DOCS_FOLDER}")
else:
    files = [f for f in DOCS_FOLDER.iterdir() if f.suffix in (".txt", ".md", ".pdf")]
    print(f"Found {len(files)} supported file(s):")
    for f in files:
        print(f"  {f.name}")

## File readers

In [ ]:
def read_txt(path):
    return path.read_text(encoding="utf-8", errors="ignore")

def read_pdf(path):
    reader = PdfReader(str(path))
    return "\n".join(page.extract_text() or "" for page in reader.pages)

def read_file(path):
    if path.suffix == ".pdf":
        return read_pdf(path)
    return read_txt(path)  # covers .txt and .md

## Summarization

For documents longer than `CHUNK_SIZE` characters, we:
1. Split into overlapping chunks
2. Summarize each chunk
3. Combine chunk summaries into one final summary

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

def call_llm(system, user):
    r = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user}
        ]
    )
    return r.choices[0].message.content

def summarize_document(text, filename):
    chunks = splitter.split_text(text)
    print(f"  {filename}: {len(chunks)} chunk(s)")

    if len(chunks) == 1:
        return call_llm(
            "Summarize the following document concisely in markdown.",
            chunks[0]
        )

    # Multi-chunk: summarize each, then combine
    chunk_summaries = [
        call_llm(
            "Summarize this excerpt concisely in 2-3 sentences.",
            chunk
        )
        for chunk in chunks
    ]
    combined = "\n\n".join(chunk_summaries)
    return call_llm(
        "You are given partial summaries of sections of a document. "
        "Write a single, cohesive markdown summary of the whole document.",
        combined
    )

## Run — summarize all documents in the folder

In [ ]:
summaries = {}  # filename -> summary text

files = [f for f in DOCS_FOLDER.iterdir() if f.suffix in (".txt", ".md", ".pdf")]

for doc_path in sorted(files):
    print(f"Summarizing: {doc_path.name}")
    text = read_file(doc_path)
    summary = summarize_document(text, doc_path.name)
    summaries[doc_path.name] = summary
    display(Markdown(f"---\n## {doc_path.name}\n\n{summary}"))

print("\nDone.")

## Optional — master summary across all documents

In [ ]:
if summaries:
    all_summaries = "\n\n".join(
        f"### {name}\n{summary}" for name, summary in summaries.items()
    )
    master = call_llm(
        "You are given summaries of multiple documents. "
        "Write a concise master summary covering all of them in markdown.",
        all_summaries
    )
    display(Markdown(f"# Master Summary\n\n{master}"))

## Optional — save all summaries to a markdown file

In [ ]:
output_path = DOCS_FOLDER / "summaries.md"

with open(output_path, "w") as f:
    for name, summary in summaries.items():
        f.write(f"## {name}\n\n{summary}\n\n---\n\n")

print(f"Saved to: {output_path}")